<a id="simple-harmonic"></a>

$$
\text{简谐振动 (Simple Harmonic Motion)} \\ \quad \\
\text{胡克定律 (Hooke's Law): 弹簧的回复力和位移成正比, 方向相反 } F = -kx \\
\text{牛顿第二定律: } m \frac{d^2 x}{dt^2} = -kx \implies \frac{d^2 x}{dt^2} + \omega_0^2 x = 0, \quad \omega_0 = \sqrt{\frac{k}{m}} \ (\text{固有角频率 natural angular frequency}) \\ \quad \\
\text{求解: 常系数线性 ODE, 猜 } x = e^{rt} \text{ (指数函数求导后形状不变)} \\
r^2 e^{rt} + \omega_0^2 e^{rt} = 0 \implies r^2 + \omega_0^2 = 0 \ (\text{特征方程}) \implies r = \pm i \omega_0 \\
x = c_1 e^{i\omega_0 t} + c_2 e^{-i\omega_0 t} \xrightarrow{\text{欧拉公式 } e^{i\theta} = \cos\theta + i\sin\theta} x = A \cos\omega_0 t + B \sin\omega_0 t \\
\text{写成振幅-相位形式: } A = C \cos\varphi, \ B = C \sin\varphi \implies x = C \cos(\omega_0 t - \varphi), \quad C = \sqrt{A^2 + B^2} \\
\text{初始条件 } x(0) = x_0, \ \dot{x}(0) = v_0 \implies A = x_0, \ B = \frac{v_0}{\omega_0} \\
\text{周期 (Period) } T = \frac{2\pi}{\omega_0} = 2\pi\sqrt{\frac{m}{k}} \text{, 和振幅无关 (等时性 isochronism)} \\ \quad \\
\text{能量守恒: 方程两边乘以 } \dot{x} \text{, 用链式法则 } \frac{d}{dt}\left(\frac{1}{2}\dot{x}^2\right) = \dot{x}\ddot{x}, \ \frac{d}{dt}\left(\frac{1}{2}x^2\right) = x\dot{x} \\
m \dot{x} \ddot{x} + k x \dot{x} = 0 \implies \frac{d}{dt}\left(\underbrace{\frac{1}{2} m \dot{x}^2}_{\text{动能}} + \underbrace{\frac{1}{2} k x^2}_{\text{势能}}\right) = 0 \implies E = \frac{1}{2} k C^2 = \text{常数}
$$

数学基础: [高阶 ODE 与特征方程](../math_basics/calculus.ipynb#ode-system) · [欧拉公式 (泰勒展开)](../math_basics/calculus.ipynb#taylor) · [链式法则](../math_basics/calculus.ipynb#derivative-rules)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def rk4_system(f, y0, t):
    # 在给定的时间网格上用 RK4 积分 y' = f(t, y)
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        k1 = f(t[n], y[n])
        k2 = f(t[n] + h / 2, y[n] + h / 2 * k1)
        k3 = f(t[n] + h / 2, y[n] + h / 2 * k2)
        k4 = f(t[n] + h, y[n] + h * k3)
        y[n + 1] = y[n] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y

m, k_spring = 0.5, 8.0
omega0 = np.sqrt(k_spring / m)
x0, v0 = 0.3, 1.0
t = np.linspace(0, 6, 3001)

shm = lambda t, s: np.array([s[1], -k_spring / m * s[0]])
y_rk4 = rk4_system(shm, [x0, v0], t)
x_analytic = x0 * np.cos(omega0 * t) + v0 / omega0 * np.sin(omega0 * t)
sol = solve_ivp(shm, [0, 6], [x0, v0], t_eval=t, rtol=1e-11, atol=1e-11)
print(f'omega0 = {omega0:.4f}, 周期 T = {2 * np.pi / omega0:.4f}')
print(f'RK4 最大误差 = {np.max(np.abs(y_rk4[:, 0] - x_analytic)):.2e}, solve_ivp 最大误差 = {np.max(np.abs(sol.y[0] - x_analytic)):.2e}')

C_amp = np.sqrt(x0**2 + (v0 / omega0)**2)
E = 0.5 * m * y_rk4[:, 1]**2 + 0.5 * k_spring * y_rk4[:, 0]**2
print(f'振幅 C = {C_amp:.6f}, 数值最大位移 = {y_rk4[:, 0].max():.6f}')
print(f'能量: 理论 kC^2/2 = {0.5 * k_spring * C_amp**2:.6f}, RK4 最小 / 最大 = {E.min():.6f} / {E.max():.6f}')

# 等时性: 不同振幅, 周期相同 (找相邻两次向上过零点)
for amp in [0.1, 1.0, 10.0]:
    s = rk4_system(shm, [amp, 0.0], t)
    up = np.where((s[:-1, 1] < 0) & (s[1:, 1] >= 0))[0]
    # 在两个网格点之间线性插值, 得到更准确的过零时刻
    t_cross = t[up] - s[up, 1] * (t[up + 1] - t[up]) / (s[up + 1, 1] - s[up, 1])
    print(f'振幅 {amp:>5}: 数值周期 = {t_cross[1] - t_cross[0]:.6f}, 2 pi / omega0 = {2 * np.pi / omega0:.6f}')

<a id="damped"></a>

$$
\text{阻尼振动 (Damped Oscillation): 加上和速度成正比的阻力 } -b\dot{x} \\
m\ddot{x} + b\dot{x} + kx = 0 \implies \ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = 0, \quad \gamma = \frac{b}{2m} \ (\text{阻尼系数}) \\ \quad \\
\text{特征方程: } r^2 + 2\gamma r + \omega_0^2 = 0 \implies r = -\gamma \pm \sqrt{\gamma^2 - \omega_0^2} \\
\text{判别式 } \gamma^2 - \omega_0^2 \text{ 的符号决定三种情况:} \\ \quad \\
\text{1. 欠阻尼 (Underdamped) } \gamma < \omega_0 \text{: } r = -\gamma \pm i\omega_d, \ \omega_d = \sqrt{\omega_0^2 - \gamma^2} \\
x = e^{-\gamma t}(A \cos\omega_d t + B \sin\omega_d t), \quad A = x_0, \ B = \frac{v_0 + \gamma x_0}{\omega_d} \\
\text{(振动, 振幅按 } e^{-\gamma t} \text{ 衰减, 频率比无阻尼时略低)} \\ \quad \\
\text{2. 临界阻尼 (Critically Damped) } \gamma = \omega_0 \text{: 重根 } r = -\gamma \text{, 只有一个解 } e^{-\gamma t} \text{, 需要第二个解} \\
\text{验证 } x = t e^{-\gamma t} \text{: } \dot{x} = (1 - \gamma t)e^{-\gamma t}, \ \ddot{x} = (\gamma^2 t - 2\gamma)e^{-\gamma t} \\
\ddot{x} + 2\gamma\dot{x} + \gamma^2 x = e^{-\gamma t}\left[(\gamma^2 t - 2\gamma) + 2\gamma(1 - \gamma t) + \gamma^2 t\right] = 0 \ \checkmark \\
x = (A + Bt)e^{-\gamma t}, \quad A = x_0, \ B = v_0 + \gamma x_0 \quad (\text{不振动, 回到平衡位置最快}) \\ \quad \\
\text{3. 过阻尼 (Overdamped) } \gamma > \omega_0 \text{: 两个负实根 } r_{1,2} = -\gamma \pm \sqrt{\gamma^2 - \omega_0^2} \\
x = c_1 e^{r_1 t} + c_2 e^{r_2 t}, \quad c_1 + c_2 = x_0, \ r_1 c_1 + r_2 c_2 = v_0 \implies c_1 = \frac{v_0 - r_2 x_0}{r_1 - r_2}, \ c_2 = x_0 - c_1 \\ \quad \\
\text{能量耗散: } \frac{dE}{dt} = m\dot{x}\ddot{x} + kx\dot{x} = \dot{x}(m\ddot{x} + kx) = \dot{x}(-b\dot{x}) = -b\dot{x}^2 \leq 0 \\
\text{品质因数 (Quality Factor) } Q = \frac{\omega_0}{2\gamma} \text{: 轻阻尼 } (\gamma \ll \omega_0) \text{ 时能量包络 } E \propto (e^{-\gamma t})^2 = e^{-2\gamma t} \text{, 一个周期损失的比例 (用 } e^{-x} \approx 1 - x \text{)} \\
\frac{\Delta E}{E} \approx 1 - e^{-2\gamma T} \approx 2\gamma \cdot \frac{2\pi}{\omega_0} = \frac{2\pi}{Q} \implies Q \approx 2\pi \frac{E}{\Delta E_{\text{每周期}}}
$$

In [ ]:
def damped_analytic(t, x0, v0, gamma, w0):
    if gamma < w0:
        wd = np.sqrt(w0**2 - gamma**2)
        return np.exp(-gamma * t) * (x0 * np.cos(wd * t) + (v0 + gamma * x0) / wd * np.sin(wd * t))
    if gamma == w0:
        return (x0 + (v0 + gamma * x0) * t) * np.exp(-gamma * t)
    r1 = -gamma + np.sqrt(gamma**2 - w0**2)
    r2 = -gamma - np.sqrt(gamma**2 - w0**2)
    c1 = (v0 - r2 * x0) / (r1 - r2)
    return c1 * np.exp(r1 * t) + (x0 - c1) * np.exp(r2 * t)

plt.figure(figsize=(4, 3))
for name, gamma in [('underdamped', 0.5), ('critical', omega0), ('overdamped', 8.0)]:
    rhs = lambda t, s, g=gamma: np.array([s[1], -2 * g * s[1] - omega0**2 * s[0]])
    y_num = rk4_system(rhs, [x0, v0], t)
    x_exact = damped_analytic(t, x0, v0, gamma, omega0)
    sol = solve_ivp(rhs, [0, 6], [x0, v0], t_eval=t, rtol=1e-11, atol=1e-12)
    print(f'{name:<12} gamma = {gamma:.3f}: RK4 误差 = {np.max(np.abs(y_num[:, 0] - x_exact)):.2e}, solve_ivp 误差 = {np.max(np.abs(sol.y[0] - x_exact)):.2e}')
    plt.plot(t, x_exact, label=f'{name} (gamma={gamma:.2f})')
plt.xlabel('t')
plt.ylabel('x')
plot_nicely()
plt.show()

# 能量耗散 dE/dt = -b v^2, 以及 Q (轻阻尼)
gamma = 0.02
b = 2 * m * gamma
rhs = lambda t, s: np.array([s[1], -2 * gamma * s[1] - omega0**2 * s[0]])
t_long = np.linspace(0, 30, 60001)
y_d = rk4_system(rhs, [x0, v0], t_long)
E_d = 0.5 * m * y_d[:, 1]**2 + 0.5 * k_spring * y_d[:, 0]**2
dE_dt = np.gradient(E_d, t_long)
print(f'dE/dt == -b v^2 ? {np.allclose(dE_dt[1:-1], -b * y_d[1:-1, 1]**2, atol=1e-4)}')
Q = omega0 / (2 * gamma)
T0 = 2 * np.pi / omega0
idx = np.searchsorted(t_long, [10, 10 + T0])
loss = (E_d[idx[0]] - E_d[idx[1]]) / E_d[idx[0]]
print(f'Q = {Q:.4f}, 一个周期的能量损失比例 = {loss:.4f}, 1 - e^(-2 gamma T) = {1 - np.exp(-2 * gamma * T0):.4f}, 近似 2 pi / Q = {2 * np.pi / Q:.4f}')

<a id="forced"></a>

$$
\text{受迫振动 (Forced Oscillation): 外力 } F_0 \cos\omega t \\
\ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = \frac{F_0}{m}\cos\omega t \\ \quad \\
\text{用复数求稳态解: 解 } \ddot{z} + 2\gamma\dot{z} + \omega_0^2 z = \frac{F_0}{m} e^{i\omega t} \text{, 因为方程是线性的且系数为实数, } x = \text{Re}(z) \\
\text{猜 } z = Z e^{i\omega t} \text{ (Z 是复振幅), 求导只是乘以 } i\omega \text{:} \\
(-\omega^2 + 2i\gamma\omega + \omega_0^2) Z = \frac{F_0}{m} \implies Z = \frac{F_0 / m}{(\omega_0^2 - \omega^2) + 2i\gamma\omega} \\
\text{复数 } a + bi = \sqrt{a^2 + b^2} \, e^{i\delta}, \ \delta = \arctan\frac{b}{a} \implies Z = A e^{-i\delta} \\
A(\omega) = \frac{F_0 / m}{\sqrt{(\omega_0^2 - \omega^2)^2 + 4\gamma^2\omega^2}}, \quad \delta = \text{atan2}(2\gamma\omega, \ \omega_0^2 - \omega^2) \\
x_{\text{稳态}} = \text{Re}(A e^{-i\delta} e^{i\omega t}) = A \cos(\omega t - \delta) \quad (\text{位移落后于外力 } \delta, \ 0 < \delta < \pi) \\
\text{完整解 = 稳态解 + 阻尼振动 (暂态, 按 } e^{-\gamma t} \text{ 衰减)} \\ \quad \\
\text{共振 (Resonance): 振幅最大, 即分母最小. 令 } u = \omega^2 \text{, } D(u) = (\omega_0^2 - u)^2 + 4\gamma^2 u \\
D'(u) = -2(\omega_0^2 - u) + 4\gamma^2 = 0 \implies u = \omega_0^2 - 2\gamma^2 \implies \omega_r = \sqrt{\omega_0^2 - 2\gamma^2} \ (\text{要求 } \gamma < \omega_0 / \sqrt{2}) \\
A_{\max} = \frac{F_0 / m}{\sqrt{4\gamma^4 + 4\gamma^2(\omega_0^2 - 2\gamma^2)}} = \frac{F_0 / m}{2\gamma\sqrt{\omega_0^2 - \gamma^2}} \\
\text{在 } \omega = \omega_0 \text{ 时 } \delta = 90^\circ \text{: 速度和外力同相, 外力做功最多} \\ \quad \\
\text{平均吸收功率: } P = \langle F \dot{x} \rangle = \frac{1}{2} F_0 \omega A \sin\delta = \frac{F_0^2}{m} \cdot \frac{\gamma\omega^2}{(\omega_0^2 - \omega^2)^2 + 4\gamma^2\omega^2} \\
\text{(用到 } \sin\delta = \frac{2\gamma\omega}{\sqrt{\cdots}} \text{), 轻阻尼时功率曲线的半高宽 } \Delta\omega \approx 2\gamma \implies Q = \frac{\omega_0}{2\gamma} \approx \frac{\omega_0}{\Delta\omega}
$$

In [ ]:
F0 = 1.0
gamma = 0.4

def amp(w):
    return F0 / m / np.sqrt((omega0**2 - w**2)**2 + 4 * gamma**2 * w**2)

def phase(w):
    return np.arctan2(2 * gamma * w, omega0**2 - w**2)

# 1. 模拟足够久, 暂态衰减完, 测量稳态振幅和相位
t_f = np.linspace(0, 60, 60001)
for w in [2.0, omega0, 6.0]:
    rhs = lambda t, s, w=w: np.array([s[1], -2 * gamma * s[1] - omega0**2 * s[0] + F0 / m * np.cos(w * t)])
    y_f = rk4_system(rhs, [0.0, 0.0], t_f)
    tail = t_f > 40
    A_num = (y_f[tail, 0].max() - y_f[tail, 0].min()) / 2
    # 相位: 用最小二乘把稳态拟合成 a cos(wt) + b sin(wt) = A cos(wt - delta)
    M = np.column_stack([np.cos(w * t_f[tail]), np.sin(w * t_f[tail])])
    a_fit, b_fit = np.linalg.lstsq(M, y_f[tail, 0], rcond=None)[0]
    print(f'omega = {w:.3f}: 振幅 数值 = {A_num:.6f}, 公式 = {amp(w):.6f}; 相位 拟合 = {np.degrees(np.arctan2(b_fit, a_fit)):.3f}°, 公式 = {np.degrees(phase(w)):.3f}°')

# 2. 共振频率
w_grid = np.linspace(0.01, 8, 200001)
w_r = np.sqrt(omega0**2 - 2 * gamma**2)
print(f'共振: 网格最大振幅在 omega = {w_grid[np.argmax(amp(w_grid))]:.4f}, 公式 sqrt(w0^2 - 2 gamma^2) = {w_r:.4f}')
print(f'最大振幅: 网格 = {amp(w_grid).max():.6f}, 公式 = {F0 / m / (2 * gamma * np.sqrt(omega0**2 - gamma**2)):.6f}')

# 3. 功率曲线半高宽 ~ 2 gamma
power = F0**2 / m * gamma * w_grid**2 / ((omega0**2 - w_grid**2)**2 + 4 * gamma**2 * w_grid**2)
half = w_grid[power >= power.max() / 2]
print(f'功率半高宽 = {half[-1] - half[0]:.4f}, 2 gamma = {2 * gamma}, Q = w0 / (2 gamma) = {omega0 / (2 * gamma):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for g_ in [0.2, 0.4, 1.0, 2.0]:
    A_g = F0 / m / np.sqrt((omega0**2 - w_grid**2)**2 + 4 * g_**2 * w_grid**2)
    axs[0].plot(w_grid, A_g, label=f'gamma={g_}')
    axs[1].plot(w_grid, np.degrees(np.arctan2(2 * g_ * w_grid, omega0**2 - w_grid**2)), label=f'gamma={g_}')
axs[0].axvline(omega0, color='k', linestyle=':')
axs[0].set_xlabel('omega')
axs[0].set_ylabel('amplitude A')
axs[0].set_title('resonance curve')
axs[1].axhline(90, color='k', linestyle=':')
axs[1].set_xlabel('omega')
axs[1].set_ylabel('phase lag delta (deg)')
plot_nicely()
plt.show()